
# Project: 7-Day Geo-Intelligence Weather Risk Modeling System (Farm Weather Risk Scorer)
### Author: aidevshakil

In [1]:
# ============================================================
# Import Libraries
# ============================================================

import os
import json
import time
import math
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from datetime import datetime, timezone
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

This cell imports all the necessary Python libraries for data manipulation, visualization, and machine learning, and sets Pandas display options for better readability.

In [2]:
# ------------------------------------------------------------
# 2. API CONFIGURATION
# ------------------------------------------------------------

API_KEY = "Your Api Key here"

BASE_URL = "https://api.weather-ai.co"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}"
}

# Free plan strategy:
# - Max 7 forecast days
# - ai=false to avoid using AI quota
# - metric units for agricultural/weather analysis
DEFAULT_DAYS = 7
DEFAULT_UNITS = "metric"
USE_AI = "false"

This cell configures the API key and base URL for the WeatherAI API, along with default parameters suitable for the free plan (7 forecast days, no AI, metric units).

In [3]:
# ------------------------------------------------------------
# 3. TARGET LOCATIONS
# ------------------------------------------------------------
locations = [
    {
        "name": "Dhaka, Bangladesh",
        "lat": 23.8103,
        "lon": 90.4125,
        "crop_type": "rice",
        "land_acres": 3.0
    },
    {
        "name": "Rajshahi, Bangladesh",
        "lat": 24.3745,
        "lon": 88.6042,
        "crop_type": "mango",
        "land_acres": 5.0
    },
    {
        "name": "Khulna, Bangladesh",
        "lat": 22.8456,
        "lon": 89.5403,
        "crop_type": "shrimp_aquaculture",
        "land_acres": 4.0
    }
]

This cell defines the target agricultural locations, including their names, geographical coordinates, crop types, and land acreage. These locations will be used to fetch weather forecasts.

In [4]:
# ------------------------------------------------------------
# 4. SAFE API REQUEST FUNCTION
# ------------------------------------------------------------

def weatherai_get(endpoint, params=None, max_retries=3, sleep_seconds=2):
    """
    Generic WeatherAI GET request wrapper with:
    - Authentication
    - Retry handling
    - Rate limit header capture
    - JSON parsing
    """
    url = f"{BASE_URL}{endpoint}"
    params = params or {}

    for attempt in range(1, max_retries + 1):
        response = requests.get(url, headers=HEADERS, params=params)

        rate_info = {
            "limit": response.headers.get("X-RateLimit-Limit"),
            "remaining": response.headers.get("X-RateLimit-Remaining"),
            "reset": response.headers.get("X-RateLimit-Reset")
        }

        if response.status_code == 200:
            try:
                return {
                    "ok": True,
                    "status_code": response.status_code,
                    "data": response.json(),
                    "rate_info": rate_info
                }
            except Exception as e:
                return {
                    "ok": False,
                    "status_code": response.status_code,
                    "error": f"JSON parse error: {e}",
                    "text": response.text,
                    "rate_info": rate_info
                }

        if response.status_code in [429, 500, 503] and attempt < max_retries:
            time.sleep(sleep_seconds * attempt)
            continue

        return {
            "ok": False,
            "status_code": response.status_code,
            "error": response.text,
            "rate_info": rate_info
        }

This cell defines a robust `weatherai_get` function for making authenticated API requests to WeatherAI, including retry logic for transient errors and parsing of rate limit information.

In [5]:
# ------------------------------------------------------------
# 5. CHECK USAGE / QUOTA
# ------------------------------------------------------------

usage_response = weatherai_get("/v1/usage")

print("Usage API response:")
print(json.dumps(usage_response, indent=2)[:3000])

Usage API response:
{
  "ok": true,
  "status_code": 200,
  "data": {
    "plan": "free",
    "period": {
      "start": "2026-06-05T09:27:46.347Z",
      "end": "2026-07-05T09:27:46.347Z",
      "requestCount": 7,
      "aiRequestCount": 1
    },
    "limits": {
      "requests": 1000,
      "aiRequests": 200,
      "maxDays": 7,
      "webhooks": false,
      "teamSeats": 1,
      "sms": false
    },
    "remaining": {
      "requests": 993,
      "aiRequests": 199
    }
  },
  "rate_info": {
    "limit": null,
    "remaining": null,
    "reset": null
  }
}


This cell calls the WeatherAI usage endpoint to check the current API quota and plan details, then prints the response to verify the API key is working and to understand usage limits.

In [6]:
# ------------------------------------------------------------
# 6. FETCH 7-DAY WEATHER FORECASTS
# ------------------------------------------------------------

raw_results = []

for loc in locations:
    params = {
        "lat": loc["lat"],
        "lon": loc["lon"],
        "days": DEFAULT_DAYS,
        "units": DEFAULT_UNITS,
        "ai": USE_AI
    }

    result = weatherai_get("/v1/weather", params=params)

    raw_results.append({
        "location": loc,
        "api_result": result
    })

    print(f"\nFetched: {loc['name']}")
    print("Status:", result["status_code"])
    print("Rate info:", result.get("rate_info"))


Fetched: Dhaka, Bangladesh
Status: 200
Rate info: {'limit': '1000', 'remaining': '992', 'reset': '1783243666'}

Fetched: Rajshahi, Bangladesh
Status: 200
Rate info: {'limit': '1000', 'remaining': '991', 'reset': '1783243666'}

Fetched: Khulna, Bangladesh
Status: 200
Rate info: {'limit': '1000', 'remaining': '990', 'reset': '1783243666'}


This cell iterates through the defined locations, fetches the 7-day weather forecast for each using the `weatherai_get` function, and stores the raw API results.

In [7]:
# ------------------------------------------------------------
# 7. INSPECT RESPONSE SHAPE
# ------------------------------------------------------------

for item in raw_results:
    print("\n==============================")
    print(item["location"]["name"])
    print("==============================")
    data = item["api_result"].get("data", {})
    print(json.dumps(data, indent=2)[:5000])
    break


Dhaka, Bangladesh
{
  "location": {
    "lat": 23.796133,
    "lon": 90.38055,
    "timezone": "Asia/Dhaka",
    "requested_lat": 23.8103,
    "requested_lon": 90.4125,
    "country": "CN"
  },
  "current": {
    "time": "2026-06-05T16:00",
    "temperature": 31.9,
    "wind_speed": 6,
    "wind_direction": 189,
    "condition_code": "95",
    "icon": "https://cdn.weather-ai.co/icons/default/95_thunderstorm_day.svg",
    "humidity": 72,
    "feels_like": 38.5,
    "uv_index": 4.45,
    "wind_gust": 13.7,
    "icon_path": "icons/weather/png/wmo-95-day-128.png"
  },
  "hourly": [
    {
      "time": "2026-06-05T00:00",
      "temperature": 30.1,
      "precipitation_probability": 0,
      "wind_speed": 4.9,
      "condition_code": "1",
      "icon": "https://cdn.weather-ai.co/icons/default/1_mainly_clear_night.svg",
      "humidity": 75,
      "feels_like": 36.2,
      "wind_gust": 10.8,
      "uv_index": 0,
      "icon_path": "icons/weather/png/wmo-1-night-128.png"
    },
    {
      "

This cell inspects the structure of the raw API response for the first location, printing a truncated JSON output to understand the available weather data fields (e.g., location, current, hourly, daily).

In [8]:
# ------------------------------------------------------------
# 8. ROBUST FORECAST PARSER: DAILY DATA + HOURLY FALLBACK
# ------------------------------------------------------------
# WeatherAI /v1/weather often returns:
# - daily: date, temp_min, temp_max
# - hourly: temperature, humidity, wind_speed, precipitation_probability, condition_code
#
# This parser:
# 1) reads daily data when available
# 2) aggregates hourly data by date
# 3) fills missing daily fields from hourly values
# 4) if no daily array exists, creates daily rows directly from hourly aggregation

def safe_get(d, keys, default=np.nan):
    if not isinstance(d, dict):
        return default
    for key in keys:
        if key in d and d[key] is not None:
            return d[key]
    return default


def get_list(data, keys):
    """Find a list at top level or inside common nested containers."""
    if not isinstance(data, dict):
        return []

    for key in keys:
        value = data.get(key)
        if isinstance(value, list):
            return value

    for parent_key in ["data", "weather", "result"]:
        nested = data.get(parent_key)
        if isinstance(nested, dict):
            for key in keys:
                value = nested.get(key)
                if isinstance(value, list):
                    return value

    return []


def most_common_non_null(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    mode = s.mode()
    return mode.iloc[0] if len(mode) else s.iloc[0]


def hourly_to_daily(hourly_list):
    hourly_rows = []

    for h in hourly_list:
        t = safe_get(h, ["time", "datetime", "date"])
        if pd.isna(t):
            continue

        hourly_rows.append({
            "date": str(t)[:10],
            "temp_hourly": safe_get(h, ["temperature", "temp"]),
            "rain_probability_hourly": safe_get(
                h,
                ["precipitation_probability", "rain_probability", "precip_probability", "pop"]
            ),
            "rain_mm_hourly": safe_get(
                h,
                ["precipitation", "rain", "rain_mm", "precip_mm", "precipitation_mm"]
            ),
            "wind_kph_hourly": safe_get(h, ["wind_speed", "wind_kph", "wind"]),
            "humidity_hourly": safe_get(h, ["humidity", "humidity_pct", "relative_humidity"]),
            "feels_like_hourly": safe_get(h, ["feels_like", "apparent_temperature"]),
            "uv_index_hourly": safe_get(h, ["uv_index"]),
            "condition_hourly": safe_get(h, ["condition_code", "condition", "weather", "summary"])
        })

    hdf = pd.DataFrame(hourly_rows)

    if hdf.empty:
        return pd.DataFrame()

    numeric_cols = [
        "temp_hourly",
        "rain_probability_hourly",
        "rain_mm_hourly",
        "wind_kph_hourly",
        "humidity_hourly",
        "feels_like_hourly",
        "uv_index_hourly"
    ]

    for col in numeric_cols:
        hdf[col] = pd.to_numeric(hdf[col], errors="coerce")

    daily_from_hourly = (
        hdf
        .groupby("date", as_index=False)
        .agg(
            temp_avg_from_hourly=("temp_hourly", "mean"),
            temp_min_from_hourly=("temp_hourly", "min"),
            temp_max_from_hourly=("temp_hourly", "max"),
            rain_probability_from_hourly=("rain_probability_hourly", "max"),
            rain_mm_from_hourly=("rain_mm_hourly", "sum"),
            wind_kph_from_hourly=("wind_kph_hourly", "mean"),
            max_wind_kph_from_hourly=("wind_kph_hourly", "max"),
            humidity_from_hourly=("humidity_hourly", "mean"),
            max_humidity_from_hourly=("humidity_hourly", "max"),
            feels_like_max_c=("feels_like_hourly", "max"),
            uv_index_max=("uv_index_hourly", "max"),
            condition_from_hourly=("condition_hourly", most_common_non_null)
        )
    )

    return daily_from_hourly


def normalize_weather_results(raw_results):
    all_rows = []

    for item in raw_results:
        loc = item["location"]
        api_result = item["api_result"]

        if not api_result.get("ok"):
            print(f"Skipping failed result: {loc['name']}")
            continue

        data = api_result["data"]

        daily_list = get_list(data, ["daily", "forecast", "days", "forecast_days", "forecastDays"])
        hourly_list = get_list(data, ["hourly", "hours", "hourly_forecast", "hourlyForecast"])

        hourly_daily = hourly_to_daily(hourly_list)

        daily_rows = []

        # Case A: use daily data when present
        for i, day in enumerate(daily_list):
            daily_rows.append({
                "location": loc["name"],
                "lat": loc["lat"],
                "lon": loc["lon"],
                "crop_type": loc["crop_type"],
                "land_acres": loc["land_acres"],
                "day_index": i + 1,
                "date": str(safe_get(day, ["date", "day", "datetime", "time"]))[:10],
                "temp_avg_c": safe_get(day, ["temp_avg", "avg_temp", "temperature_avg", "temperature", "temp"]),
                "temp_min_c": safe_get(day, ["temp_min", "min_temp", "temperature_min"]),
                "temp_max_c": safe_get(day, ["temp_max", "max_temp", "temperature_max"]),
                "rain_mm": safe_get(day, ["rain_mm", "rain", "precipitation", "precip_mm"]),
                "rain_probability": safe_get(day, ["rain_probability", "precipitation_probability", "pop"]),
                "wind_kph": safe_get(day, ["wind_kph", "wind_speed", "wind"]),
                "humidity_pct": safe_get(day, ["humidity", "humidity_pct", "relative_humidity"]),
                "condition": safe_get(day, ["condition", "condition_code", "summary", "weather"])
            })

        daily_df = pd.DataFrame(daily_rows)

        # Case B: if daily data is absent, build rows directly from hourly aggregates
        if daily_df.empty and not hourly_daily.empty:
            daily_df = hourly_daily.copy()
            daily_df["location"] = loc["name"]
            daily_df["lat"] = loc["lat"]
            daily_df["lon"] = loc["lon"]
            daily_df["crop_type"] = loc["crop_type"]
            daily_df["land_acres"] = loc["land_acres"]
            daily_df["day_index"] = range(1, len(daily_df) + 1)

            daily_df["temp_avg_c"] = daily_df["temp_avg_from_hourly"]
            daily_df["temp_min_c"] = daily_df["temp_min_from_hourly"]
            daily_df["temp_max_c"] = daily_df["temp_max_from_hourly"]
            daily_df["rain_probability"] = daily_df["rain_probability_from_hourly"]
            daily_df["rain_mm"] = daily_df["rain_mm_from_hourly"]
            daily_df["wind_kph"] = daily_df["wind_kph_from_hourly"]
            daily_df["humidity_pct"] = daily_df["humidity_from_hourly"]
            daily_df["condition"] = daily_df["condition_from_hourly"]

        # Case C: daily exists, enrich/fill it from hourly aggregates
        elif not daily_df.empty and not hourly_daily.empty:
            daily_df = daily_df.merge(hourly_daily, on="date", how="left")

            fill_pairs = {
                "temp_avg_c": "temp_avg_from_hourly",
                "temp_min_c": "temp_min_from_hourly",
                "temp_max_c": "temp_max_from_hourly",
                "rain_probability": "rain_probability_from_hourly",
                "rain_mm": "rain_mm_from_hourly",
                "wind_kph": "wind_kph_from_hourly",
                "humidity_pct": "humidity_from_hourly",
                "condition": "condition_from_hourly"
            }

            for target, source in fill_pairs.items():
                if source in daily_df.columns:
                    daily_df[target] = daily_df[target].fillna(daily_df[source])

        if daily_df.empty:
            print(f"No usable daily/hourly forecast found for {loc['name']}")
            continue

        all_rows.append(daily_df)

    if not all_rows:
        return pd.DataFrame()

    final_df = pd.concat(all_rows, ignore_index=True)

    numeric_cols = [
        "temp_avg_c", "temp_min_c", "temp_max_c",
        "rain_mm", "rain_probability",
        "wind_kph", "humidity_pct",
        "land_acres",
        "feels_like_max_c", "uv_index_max"
    ]

    for col in numeric_cols:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

    # Keep clean main columns first, but preserve extra diagnostic columns too.
    main_cols = [
        "location", "lat", "lon", "crop_type", "land_acres", "day_index", "date",
        "temp_avg_c", "temp_min_c", "temp_max_c",
        "rain_mm", "rain_probability", "wind_kph", "humidity_pct",
        "condition", "feels_like_max_c", "uv_index_max"
    ]
    ordered_cols = [c for c in main_cols if c in final_df.columns] + [
        c for c in final_df.columns if c not in main_cols
    ]

    return final_df[ordered_cols]


weather_df = normalize_weather_results(raw_results)

weather_df[
    [
        "location", "date", "temp_avg_c", "temp_min_c", "temp_max_c",
        "rain_mm", "rain_probability", "wind_kph", "humidity_pct", "condition"
    ]
]


,location,date,temp_avg_c,temp_min_c,temp_max_c,rain_mm,rain_probability,wind_kph,humidity_pct,condition
0,"Dhaka, Bangladesh",2026-06-05,29.483333,25.7,31.9,0.0,45,5.208333,82.500000,95
1,"Dhaka, Bangladesh",2026-06-06,28.925000,26.9,32.2,0.0,82,4.845833,87.791667,95
2,"Dhaka, Bangladesh",2026-06-07,28.612500,26.4,31.2,0.0,78,8.020833,86.791667,95
3,"Dhaka, Bangladesh",2026-06-08,28.100000,25.0,31.5,0.0,62,8.133333,86.791667,95
4,"Dhaka, Bangladesh",2026-06-09,29.504167,26.6,32.1,0.0,65,11.962500,86.625000,95
5,"Dhaka, Bangladesh",2026-06-10,28.887500,27.1,31.0,0.0,65,11.008333,88.833333,95
6,"Dhaka, Bangladesh",2026-06-11,29.795833,27.5,32.3,0.0,74,10.154167,83.750000,95
7,"Rajshahi, Bangladesh",2026-06-05,30.333333,26.8,34.0,0.0,76,7.250000,77.916667,95
8,"Rajshahi, Bangladesh",2026-06-06,28.500000,25.3,33.2,0.0,88,6.250000,88.000000,95
9,"Rajshahi, Bangladesh",2026-06-07,28.575000,27.1,32.5,0.0,65,5.675000,90.708333,95


This cell defines a comprehensive data parsing function `normalize_weather_results` that extracts daily weather data from the raw API responses. It prioritizes daily data when available, aggregates hourly data to create daily summaries if needed, and fills in missing daily information from hourly aggregates. This ensures a consistent daily forecast DataFrame.

In [9]:
# ------------------------------------------------------------
# 9. DATA QUALITY ANALYSIS
# ------------------------------------------------------------

print("Rows:", len(weather_df))
print("Columns:", weather_df.columns.tolist())

missing_report = (
    weather_df.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_pct"]
missing_report


Rows: 21
Columns: ['location', 'lat', 'lon', 'crop_type', 'land_acres', 'day_index', 'date', 'temp_avg_c', 'temp_min_c', 'temp_max_c', 'rain_mm', 'rain_probability', 'wind_kph', 'humidity_pct', 'condition', 'feels_like_max_c', 'uv_index_max', 'temp_avg_from_hourly', 'temp_min_from_hourly', 'temp_max_from_hourly', 'rain_probability_from_hourly', 'rain_mm_from_hourly', 'wind_kph_from_hourly', 'max_wind_kph_from_hourly', 'humidity_from_hourly', 'max_humidity_from_hourly', 'condition_from_hourly']


,column,missing_pct
0,location,0.0
1,lat,0.0
2,lon,0.0
3,crop_type,0.0
4,land_acres,0.0
5,day_index,0.0
6,date,0.0
7,temp_avg_c,0.0
8,temp_min_c,0.0
9,temp_max_c,0.0


This cell performs a data quality analysis on the `weather_df` DataFrame, reporting the number of rows and columns, and crucially, calculating the percentage of missing values for each column. This helps identify any data completeness issues.

In [10]:
# ------------------------------------------------------------
# 10. IMPUTE MISSING VALUES SAFELY
# ------------------------------------------------------------
# This cell prevents NaN values from breaking scoring, charts, and clustering.

df = weather_df.copy()

required_numeric_cols = [
    "temp_avg_c", "temp_min_c", "temp_max_c",
    "rain_mm", "rain_probability", "wind_kph", "humidity_pct"
]

for col in required_numeric_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Temperature fallback:
# If average temperature is missing, use mean of min/max.
df["temp_avg_c"] = df["temp_avg_c"].fillna(
    df[["temp_min_c", "temp_max_c"]].mean(axis=1)
)

# If min/max are missing, use average temperature.
df["temp_min_c"] = df["temp_min_c"].fillna(df["temp_avg_c"])
df["temp_max_c"] = df["temp_max_c"].fillna(df["temp_avg_c"])

# Rainfall amount is often unavailable in the free response.
# Keep it as 0, but risk modeling will also use rain_probability.
df["rain_mm"] = df["rain_mm"].fillna(0)
df["rain_probability"] = df["rain_probability"].fillna(0)

# Wind/humidity fallbacks.
# Use 0 only after trying the location-level median.
for col in ["wind_kph", "humidity_pct"]:
    df[col] = df.groupby("location")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median()).fillna(0)

if "condition" not in df.columns:
    df["condition"] = "unknown"
else:
    df["condition"] = df["condition"].fillna("unknown").astype(str)

df[
    [
        "location", "date", "temp_avg_c", "temp_min_c", "temp_max_c",
        "rain_mm", "rain_probability", "wind_kph", "humidity_pct", "condition"
    ]
]


,location,date,temp_avg_c,temp_min_c,temp_max_c,rain_mm,rain_probability,wind_kph,humidity_pct,condition
0,"Dhaka, Bangladesh",2026-06-05,29.483333,25.7,31.9,0.0,45,5.208333,82.500000,95
1,"Dhaka, Bangladesh",2026-06-06,28.925000,26.9,32.2,0.0,82,4.845833,87.791667,95
2,"Dhaka, Bangladesh",2026-06-07,28.612500,26.4,31.2,0.0,78,8.020833,86.791667,95
3,"Dhaka, Bangladesh",2026-06-08,28.100000,25.0,31.5,0.0,62,8.133333,86.791667,95
4,"Dhaka, Bangladesh",2026-06-09,29.504167,26.6,32.1,0.0,65,11.962500,86.625000,95
5,"Dhaka, Bangladesh",2026-06-10,28.887500,27.1,31.0,0.0,65,11.008333,88.833333,95
6,"Dhaka, Bangladesh",2026-06-11,29.795833,27.5,32.3,0.0,74,10.154167,83.750000,95
7,"Rajshahi, Bangladesh",2026-06-05,30.333333,26.8,34.0,0.0,76,7.250000,77.916667,95
8,"Rajshahi, Bangladesh",2026-06-06,28.500000,25.3,33.2,0.0,88,6.250000,88.000000,95
9,"Rajshahi, Bangladesh",2026-06-07,28.575000,27.1,32.5,0.0,65,5.675000,90.708333,95


This cell handles missing values in the `weather_df` DataFrame. It imputes missing temperature values using the mean of min/max or the average temperature, sets missing rain amounts and probabilities to zero, and fills missing wind and humidity values using location-level medians or zeros. This ensures a clean dataset for subsequent risk modeling.

In [11]:
# ------------------------------------------------------------
# 11. EXPERT FEATURE ENGINEERING
# ------------------------------------------------------------

def crop_thresholds(crop_type):
    """
    Expert-defined crop thresholds for prototype risk modeling.
    These are modeling assumptions, not universal agronomic constants.
    """
    crop_type = str(crop_type).lower()

    base = {
        "heat_stress_c": 32,
        "cold_stress_c": 8,
        "heavy_rain_mm": 25,
        "very_heavy_rain_mm": 50,
        "rain_probability_warning": 50,
        "rain_probability_high": 80,
        "wind_risk_kph": 35,
        "humidity_disease_pct": 85
    }

    crop_specific = {
        "rice": {
            "heat_stress_c": 35,
            "cold_stress_c": 12,
            "heavy_rain_mm": 40,
            "rain_probability_warning": 55,
            "humidity_disease_pct": 85
        },
        "mango": {
            "heat_stress_c": 34,
            "cold_stress_c": 10,
            "heavy_rain_mm": 25,
            "rain_probability_warning": 45,
            "humidity_disease_pct": 80
        },
        "shrimp_aquaculture": {
            "heat_stress_c": 34,
            "cold_stress_c": 18,
            "heavy_rain_mm": 30,
            "rain_probability_warning": 50,
            "humidity_disease_pct": 82
        },
        "maize": {
            "heat_stress_c": 33,
            "cold_stress_c": 10,
            "heavy_rain_mm": 30,
            "humidity_disease_pct": 80
        },
        "tea": {
            "heat_stress_c": 30,
            "cold_stress_c": 7,
            "heavy_rain_mm": 35,
            "humidity_disease_pct": 88
        }
    }

    base.update(crop_specific.get(crop_type, {}))
    return base


def clipped_score(value, low, high):
    """
    Converts a metric into a 0-100 risk score.
    Below low = 0 risk, above high = 100 risk.
    """
    if pd.isna(value):
        return 0
    if high == low:
        return 0
    return max(0, min(100, 100 * (value - low) / (high - low)))


feature_rows = []

for _, row in df.iterrows():
    th = crop_thresholds(row["crop_type"])

    heat_score = clipped_score(row["temp_max_c"], th["heat_stress_c"], th["heat_stress_c"] + 8)
    cold_score = clipped_score(th["cold_stress_c"] - row["temp_min_c"], 0, 8)

    # Rain risk uses BOTH amount and probability.
    # This is important because the free API response often gives probability but not actual mm.
    rain_amount_score = clipped_score(row["rain_mm"], th["heavy_rain_mm"], th["very_heavy_rain_mm"])
    rain_probability_score = clipped_score(
        row["rain_probability"],
        th["rain_probability_warning"],
        th["rain_probability_high"]
    )
    rain_score = max(rain_amount_score, rain_probability_score)

    wind_score = clipped_score(row["wind_kph"], th["wind_risk_kph"], th["wind_risk_kph"] + 35)
    humidity_score = clipped_score(row["humidity_pct"], th["humidity_disease_pct"], 100)

    composite_risk = (
        0.30 * rain_score +
        0.25 * heat_score +
        0.15 * cold_score +
        0.15 * wind_score +
        0.15 * humidity_score
    )

    feature_rows.append({
        **row.to_dict(),
        "heat_score": round(heat_score, 2),
        "cold_score": round(cold_score, 2),
        "rain_amount_score": round(rain_amount_score, 2),
        "rain_probability_score": round(rain_probability_score, 2),
        "rain_score": round(rain_score, 2),
        "wind_score": round(wind_score, 2),
        "humidity_disease_score": round(humidity_score, 2),
        "composite_risk_score": round(composite_risk, 2)
    })

risk_df = pd.DataFrame(feature_rows)

def risk_level(score):
    if score >= 75:
        return "Extreme"
    if score >= 55:
        return "High"
    if score >= 30:
        return "Medium"
    return "Low"

risk_df["risk_level"] = risk_df["composite_risk_score"].apply(risk_level)

risk_df[
    [
        "location", "date", "crop_type",
        "rain_probability", "rain_probability_score",
        "temp_max_c", "heat_score",
        "humidity_pct", "humidity_disease_score",
        "composite_risk_score", "risk_level"
    ]
]


,location,date,crop_type,rain_probability,rain_probability_score,temp_max_c,heat_score,humidity_pct,humidity_disease_score,composite_risk_score,risk_level
0,"Dhaka, Bangladesh",2026-06-05,rice,45,0.00,31.9,0.00,82.500000,0.00,0.00,Low
1,"Dhaka, Bangladesh",2026-06-06,rice,82,100.00,32.2,0.00,87.791667,18.61,32.79,Medium
2,"Dhaka, Bangladesh",2026-06-07,rice,78,92.00,31.2,0.00,86.791667,11.94,29.39,Low
3,"Dhaka, Bangladesh",2026-06-08,rice,62,28.00,31.5,0.00,86.791667,11.94,10.19,Low
4,"Dhaka, Bangladesh",2026-06-09,rice,65,40.00,32.1,0.00,86.625000,10.83,13.62,Low
5,"Dhaka, Bangladesh",2026-06-10,rice,65,40.00,31.0,0.00,88.833333,25.56,15.83,Low
6,"Dhaka, Bangladesh",2026-06-11,rice,74,76.00,32.3,0.00,83.750000,0.00,22.80,Low
7,"Rajshahi, Bangladesh",2026-06-05,mango,76,88.57,34.0,0.00,77.916667,0.00,26.57,Low
8,"Rajshahi, Bangladesh",2026-06-06,mango,88,100.00,33.2,0.00,88.000000,40.00,36.00,Medium
9,"Rajshahi, Bangladesh",2026-06-07,mango,65,57.14,32.5,0.00,90.708333,53.54,25.17,Low


This cell implements expert feature engineering to calculate various risk scores (heat, cold, rain, wind, humidity) based on crop-specific thresholds. It defines a `crop_thresholds` function for different crop types and a `clipped_score` function to normalize risk metrics. Finally, it calculates a `composite_risk_score` and assigns an overall `risk_level` for each forecast day and location.

In [12]:
# ------------------------------------------------------------
# 12. RECOMMENDATION ENGINE
# ------------------------------------------------------------

def generate_recommendations(row):
    recs = []

    if row["rain_score"] >= 70:
        recs.append("High rainfall risk: avoid fertilizer or pesticide application before rain.")
        recs.append("Inspect drainage channels and low-lying parts of the field.")
    elif row["rain_score"] >= 30:
        recs.append("Moderate rainfall: schedule field work during drier windows.")

    if row["heat_score"] >= 70:
        recs.append("Heat stress likely: irrigate early morning or late evening if water is available.")
        recs.append("Avoid transplanting or heavy crop handling during peak heat.")
    elif row["heat_score"] >= 30:
        recs.append("Moderate heat stress: monitor young plants and exposed soil moisture.")

    if row["cold_score"] >= 70:
        recs.append("Cold stress risk: protect seedlings and avoid planting sensitive crops.")
    elif row["cold_score"] >= 30:
        recs.append("Mild cold stress: monitor germination and early growth.")

    if row["wind_score"] >= 70:
        recs.append("Strong wind risk: avoid spraying chemicals and secure light structures.")
    elif row["wind_score"] >= 30:
        recs.append("Moderate wind: use caution with spraying and irrigation uniformity.")

    if row["humidity_disease_score"] >= 70:
        recs.append("High humidity disease pressure: scout for fungal symptoms.")
    elif row["humidity_disease_score"] >= 30:
        recs.append("Moderate disease pressure: improve spacing, pruning, and field ventilation where possible.")

    if not recs:
        recs.append("Low weather risk: suitable window for normal field operations.")

    return recs


risk_df["recommendations"] = risk_df.apply(generate_recommendations, axis=1)
risk_df[["location", "date", "crop_type", "composite_risk_score", "risk_level", "recommendations"]]

,location,date,crop_type,composite_risk_score,risk_level,recommendations
0,"Dhaka, Bangladesh",2026-06-05,rice,0.00,Low,[Low weather risk: suitable window for normal ...
1,"Dhaka, Bangladesh",2026-06-06,rice,32.79,Medium,[High rainfall risk: avoid fertilizer or pesti...
2,"Dhaka, Bangladesh",2026-06-07,rice,29.39,Low,[High rainfall risk: avoid fertilizer or pesti...
3,"Dhaka, Bangladesh",2026-06-08,rice,10.19,Low,[Low weather risk: suitable window for normal ...
4,"Dhaka, Bangladesh",2026-06-09,rice,13.62,Low,[Moderate rainfall: schedule field work during...
5,"Dhaka, Bangladesh",2026-06-10,rice,15.83,Low,[Moderate rainfall: schedule field work during...
6,"Dhaka, Bangladesh",2026-06-11,rice,22.80,Low,[High rainfall risk: avoid fertilizer or pesti...
7,"Rajshahi, Bangladesh",2026-06-05,mango,26.57,Low,[High rainfall risk: avoid fertilizer or pesti...
8,"Rajshahi, Bangladesh",2026-06-06,mango,36.00,Medium,[High rainfall risk: avoid fertilizer or pesti...
9,"Rajshahi, Bangladesh",2026-06-07,mango,25.17,Low,[Moderate rainfall: schedule field work during...


This cell develops a rule-based recommendation engine. The `generate_recommendations` function analyzes the calculated risk scores for each weather component and provides tailored operational advice for farmers based on the severity of each risk (e.g., high rainfall risk leads to specific actions).

In [13]:
# ------------------------------------------------------------
# 13. LOCATION-LEVEL SUMMARY
# ------------------------------------------------------------

summary_df = (
    risk_df
    .groupby(["location", "crop_type", "lat", "lon"], as_index=False)
    .agg(
        avg_risk_score=("composite_risk_score", "mean"),
        max_risk_score=("composite_risk_score", "max"),
        high_or_extreme_days=("risk_level", lambda x: sum(v in ["High", "Extreme"] for v in x)),
        total_rain_mm=("rain_mm", "sum"),
        max_temp_c=("temp_max_c", "max"),
        min_temp_c=("temp_min_c", "min"),
        max_wind_kph=("wind_kph", "max"),
        avg_humidity_pct=("humidity_pct", "mean")
    )
)

summary_df["avg_risk_score"] = summary_df["avg_risk_score"].round(2)
summary_df["avg_humidity_pct"] = summary_df["avg_humidity_pct"].round(2)
summary_df["overall_risk_level"] = summary_df["avg_risk_score"].apply(risk_level)

summary_df

,location,crop_type,lat,lon,avg_risk_score,max_risk_score,high_or_extreme_days,total_rain_mm,max_temp_c,min_temp_c,max_wind_kph,avg_humidity_pct,overall_risk_level
0,"Dhaka, Bangladesh",rice,23.8103,90.4125,17.80,32.79,0,0.0,32.3,25.0,11.962500,86.15,Low
1,"Khulna, Bangladesh",shrimp_aquaculture,22.8456,89.5403,25.91,36.01,0,0.0,36.5,25.4,12.783333,89.64,Low
2,"Rajshahi, Bangladesh",mango,24.3745,88.6042,18.53,36.00,0,0.0,36.6,25.3,10.845833,84.12,Low


This cell generates a location-level summary DataFrame (`summary_df`). It aggregates the daily risk data to calculate overall metrics like average and maximum risk scores, number of high/extreme risk days, total rainfall, temperature extremes, max wind, and average humidity for each location. It also assigns an `overall_risk_level`.

In [14]:
# ------------------------------------------------------------
# 14. VISUAL ANALYSIS: RISK TREND
# ------------------------------------------------------------

fig = px.line(
    risk_df,
    x="day_index",
    y="composite_risk_score",
    color="location",
    markers=True,
    title="7-Day Composite Weather Risk Forecast by Location",
    labels={
        "day_index": "Forecast Day",
        "composite_risk_score": "Composite Risk Score"
    }
)

fig.show()

This cell generates an interactive line plot using Plotly Express, visualizing the 7-day composite weather risk score trend for each location. This helps to quickly identify which days pose the highest overall risk.

In [15]:
# ------------------------------------------------------------
# 15. VISUAL ANALYSIS: RISK COMPONENTS
# ------------------------------------------------------------

component_cols = [
    "rain_score",
    "heat_score",
    "cold_score",
    "wind_score",
    "humidity_disease_score"
]

component_long = risk_df.melt(
    id_vars=["location", "day_index", "date", "crop_type"],
    value_vars=component_cols,
    var_name="risk_component",
    value_name="score"
)

fig = px.bar(
    component_long,
    x="day_index",
    y="score",
    color="risk_component",
    facet_row="location",
    title="Daily Risk Component Breakdown",
    labels={
        "day_index": "Forecast Day",
        "score": "Risk Component Score"
    }
)

fig.show()

This cell creates an interactive bar chart using Plotly Express to show the breakdown of individual risk components (rain, heat, cold, wind, humidity) for each day and location. This allows for a detailed understanding of what weather factors contribute most to the overall risk.

In [16]:
# ------------------------------------------------------------
# 16. VISUAL ANALYSIS: RAINFALL VS RISK
# ------------------------------------------------------------

plot_df = risk_df.copy()

plot_df["rain_mm"] = pd.to_numeric(plot_df["rain_mm"], errors="coerce").fillna(0)
plot_df["composite_risk_score"] = pd.to_numeric(
    plot_df["composite_risk_score"], errors="coerce"
).fillna(0)

# Plotly marker size cannot contain NaN or negative values
plot_df["wind_kph_plot"] = pd.to_numeric(
    plot_df["wind_kph"], errors="coerce"
).fillna(0)

# If all wind values are 0/missing, use fixed-size markers
if plot_df["wind_kph_plot"].sum() == 0:
    fig = px.scatter(
        plot_df,
        x="rain_mm",
        y="composite_risk_score",
        color="location",
        hover_data=["crop_type", "date", "temp_max_c", "humidity_pct", "risk_level"],
        title="Rainfall and Composite Farm Risk",
        labels={
            "rain_mm": "Rainfall mm",
            "composite_risk_score": "Composite Risk Score"
        }
    )
else:
    fig = px.scatter(
        plot_df,
        x="rain_mm",
        y="composite_risk_score",
        color="location",
        size="wind_kph_plot",
        hover_data=["crop_type", "date", "temp_max_c", "humidity_pct", "risk_level"],
        title="Rainfall, Wind, and Composite Farm Risk",
        labels={
            "rain_mm": "Rainfall mm",
            "composite_risk_score": "Composite Risk Score",
            "wind_kph_plot": "Wind speed kph"
        }
    )

fig.show()

This cell visualizes the relationship between rainfall, wind speed, and composite farm risk using an interactive scatter plot. The size of the markers represents wind speed, providing insights into how these factors combine to influence risk levels.

In [17]:
# ------------------------------------------------------------
# 17. UNSUPERVISED WEATHER PATTERN CLUSTERING
# ------------------------------------------------------------
# This adds a predictive/modeling component without requiring paid endpoints.

cluster_features = [
    "temp_avg_c",
    "temp_min_c",
    "temp_max_c",
    "rain_mm",
    "wind_kph",
    "humidity_pct",
    "composite_risk_score"
]

cluster_df = risk_df.dropna(subset=cluster_features).copy()

if len(cluster_df) >= 3:
    scaler = MinMaxScaler()
    X = scaler.fit_transform(cluster_df[cluster_features])

    k = min(3, len(cluster_df))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_df["weather_regime"] = kmeans.fit_predict(X)

    risk_df = risk_df.merge(
        cluster_df[["location", "day_index", "weather_regime"]],
        on=["location", "day_index"],
        how="left"
    )

    display(risk_df[["location", "day_index", "date", "weather_regime", "composite_risk_score", "risk_level"]])

    fig = px.scatter(
        risk_df,
        x="rain_mm",
        y="temp_max_c",
        color="weather_regime",
        size="composite_risk_score",
        hover_data=["location", "date", "risk_level", "wind_kph", "humidity_pct"],
        title="Discovered Weather Regimes from Free Forecast Data"
    )

    fig.show()
else:
    print("Not enough rows for clustering.")

,location,day_index,date,weather_regime,composite_risk_score,risk_level
0,"Dhaka, Bangladesh",1,2026-06-05,1,0.00,Low
1,"Dhaka, Bangladesh",2,2026-06-06,0,32.79,Medium
2,"Dhaka, Bangladesh",3,2026-06-07,0,29.39,Low
3,"Dhaka, Bangladesh",4,2026-06-08,0,10.19,Low
4,"Dhaka, Bangladesh",5,2026-06-09,2,13.62,Low
5,"Dhaka, Bangladesh",6,2026-06-10,2,15.83,Low
6,"Dhaka, Bangladesh",7,2026-06-11,2,22.80,Low
7,"Rajshahi, Bangladesh",1,2026-06-05,1,26.57,Low
8,"Rajshahi, Bangladesh",2,2026-06-06,0,36.00,Medium
9,"Rajshahi, Bangladesh",3,2026-06-07,0,25.17,Low


This cell applies unsupervised learning (KMeans clustering) to identify distinct weather patterns or 'regimes' based on a selection of numerical features. It then visualizes these clusters, helping to categorize days into different weather scenarios (e.g., hot and dry, cool and wet).

In [18]:
# ------------------------------------------------------------
# 18. EXPERT ANALYTICAL REPORT GENERATOR
# ------------------------------------------------------------

def make_location_report(location_name):
    loc_df = risk_df[risk_df["location"] == location_name].copy()
    loc_summary = summary_df[summary_df["location"] == location_name].iloc[0]

    worst_day = loc_df.sort_values("composite_risk_score", ascending=False).iloc[0]

    report = f"""
============================================================
WEATHERAI FREE PLAN GEO-INTELLIGENCE REPORT
Location: {location_name}
Crop Type: {loc_summary['crop_type']}
Forecast Horizon: 7 days
============================================================

1. EXECUTIVE SUMMARY
The average composite weather risk score is {loc_summary['avg_risk_score']} / 100,
classified as {loc_summary['overall_risk_level']} risk. The maximum daily risk score
is {loc_summary['max_risk_score']} / 100. There are {loc_summary['high_or_extreme_days']}
High or Extreme risk days in the 7-day forecast window.

2. KEY WEATHER DRIVERS
Total expected rainfall: {loc_summary['total_rain_mm']} mm
Maximum forecast temperature: {loc_summary['max_temp_c']} °C
Minimum forecast temperature: {loc_summary['min_temp_c']} °C
Maximum wind speed: {loc_summary['max_wind_kph']} kph
Average humidity: {loc_summary['avg_humidity_pct']}%

3. HIGHEST-RISK DAY
Forecast day: {worst_day['day_index']}
Date: {worst_day['date']}
Risk score: {worst_day['composite_risk_score']} / 100
Risk level: {worst_day['risk_level']}

Risk component scores:
- Rain risk: {round(worst_day['rain_score'], 2)}
- Heat risk: {round(worst_day['heat_score'], 2)}
- Cold risk: {round(worst_day['cold_score'], 2)}
- Wind risk: {round(worst_day['wind_score'], 2)}
- Humidity disease pressure: {round(worst_day['humidity_disease_score'], 2)}

4. OPERATIONAL RECOMMENDATIONS
"""

    for rec in worst_day["recommendations"]:
        report += f"- {rec}\n"

    report += """
5. MODELING NOTES
This notebook intentionally uses only WeatherAI Free plan-compatible endpoints.
The intelligence layer is implemented locally through feature engineering,
crop-specific risk thresholds, weighted risk scoring, unsupervised clustering,
and rule-based recommendation generation.

6. LIMITATIONS
The scoring model is a decision-support prototype, not a certified agronomic
advisory system. Thresholds should be calibrated using local crop calendars,
soil data, historical yields, pest records, and farmer feedback.
"""

    return report


for location_name in risk_df["location"].unique():
    print(make_location_report(location_name))


WEATHERAI FREE PLAN GEO-INTELLIGENCE REPORT
Location: Dhaka, Bangladesh
Crop Type: rice
Forecast Horizon: 7 days

1. EXECUTIVE SUMMARY
The average composite weather risk score is 17.8 / 100,
classified as Low risk. The maximum daily risk score
is 32.79 / 100. There are 0
High or Extreme risk days in the 7-day forecast window.

2. KEY WEATHER DRIVERS
Total expected rainfall: 0.0 mm
Maximum forecast temperature: 32.3 °C
Minimum forecast temperature: 25.0 °C
Maximum wind speed: 11.9625 kph
Average humidity: 86.15%

3. HIGHEST-RISK DAY
Forecast day: 2
Date: 2026-06-06
Risk score: 32.79 / 100
Risk level: Medium

Risk component scores:
- Rain risk: 100.0
- Heat risk: 0.0
- Cold risk: 0
- Wind risk: 0
- Humidity disease pressure: 18.61

4. OPERATIONAL RECOMMENDATIONS
- High rainfall risk: avoid fertilizer or pesticide application before rain.
- Inspect drainage channels and low-lying parts of the field.

5. MODELING NOTES
This notebook intentionally uses only WeatherAI Free plan-compatible e

This cell defines a function `make_location_report` that generates a detailed, human-readable analytical report for each location. The report includes an executive summary, key weather drivers, the highest-risk day with component scores, and operational recommendations, providing a comprehensive overview of the forecast and its implications.

In [19]:
# ------------------------------------------------------------
# 19. EXPORT RESULTS
# ------------------------------------------------------------

risk_df.to_csv("weatherai_daily_risk_analysis.csv", index=False)
summary_df.to_csv("weatherai_location_summary.csv", index=False)

print("Exported:")
print("- weatherai_daily_risk_analysis.csv")
print("- weatherai_location_summary.csv")

Exported:
- weatherai_daily_risk_analysis.csv
- weatherai_location_summary.csv


This cell exports the `risk_df` (daily risk analysis) and `summary_df` (location-level summary) DataFrames to CSV files. This allows for easy sharing and further analysis of the results outside the notebook environment.

In [20]:
# ------------------------------------------------------------
# 20. OPTIONAL: SINGLE LOCATION JSON OUTPUT FOR APP/API DEMO
# ------------------------------------------------------------

def build_api_style_output(location_name):
    loc_df = risk_df[risk_df["location"] == location_name].copy()
    loc_summary = summary_df[summary_df["location"] == location_name].iloc[0]

    daily = []

    for _, row in loc_df.iterrows():
        daily.append({
            "day_index": int(row["day_index"]),
            "date": str(row["date"]),
            "risk_score": float(row["composite_risk_score"]),
            "risk_level": row["risk_level"],
            "weather": {
                "rain_mm": None if pd.isna(row["rain_mm"]) else float(row["rain_mm"]),
                "temp_min_c": None if pd.isna(row["temp_min_c"]) else float(row["temp_min_c"]),
                "temp_max_c": None if pd.isna(row["temp_max_c"]) else float(row["temp_max_c"]),
                "wind_kph": None if pd.isna(row["wind_kph"]) else float(row["wind_kph"]),
                "humidity_pct": None if pd.isna(row["humidity_pct"]) else float(row["humidity_pct"])
            },
            "risk_components": {
                "rain": float(row["rain_score"]),
                "heat": float(row["heat_score"]),
                "cold": float(row["cold_score"]),
                "wind": float(row["wind_score"]),
                "humidity_disease": float(row["humidity_disease_score"])
            },
            "recommendations": row["recommendations"]
        })

    output = {
        "location": location_name,
        "crop_type": loc_summary["crop_type"],
        "forecast_days": 7,
        "overall": {
            "avg_risk_score": float(loc_summary["avg_risk_score"]),
            "max_risk_score": float(loc_summary["max_risk_score"]),
            "risk_level": loc_summary["overall_risk_level"],
            "high_or_extreme_days": int(loc_summary["high_or_extreme_days"])
        },
        "daily_forecast_risk": daily
    }

    return output


demo_output = build_api_style_output(risk_df["location"].unique()[0])
print(json.dumps(demo_output, indent=2))

{
  "location": "Dhaka, Bangladesh",
  "crop_type": "rice",
  "forecast_days": 7,
  "overall": {
    "avg_risk_score": 17.8,
    "max_risk_score": 32.79,
    "risk_level": "Low",
    "high_or_extreme_days": 0
  },
  "daily_forecast_risk": [
    {
      "day_index": 1,
      "date": "2026-06-05",
      "risk_score": 0.0,
      "risk_level": "Low",
      "weather": {
        "rain_mm": 0.0,
        "temp_min_c": 25.7,
        "temp_max_c": 31.9,
        "wind_kph": 5.208333333333333,
        "humidity_pct": 82.5
      },
      "risk_components": {
        "rain": 0.0,
        "heat": 0.0,
        "cold": 0.0,
        "wind": 0.0,
        "humidity_disease": 0.0
      },
      "recommendations": [
        "Low weather risk: suitable window for normal field operations."
      ]
    },
    {
      "day_index": 2,
      "date": "2026-06-06",
      "risk_score": 32.79,
      "risk_level": "Medium",
      "weather": {
        "rain_mm": 0.0,
        "temp_min_c": 26.9,
        "temp_max_c": 32

This cell provides an optional function `build_api_style_output` to format the risk analysis results for a single location into a structured JSON output. This is useful for demonstrating how the generated intelligence could be consumed by an external application or API, including daily forecasts, risk scores, components, and recommendations.